# Company Research & Brochure Assistant

Give it a company name and website URL. It scrapes the site, picks out the pages worth reading (About, Careers, etc.), writes a short brochure about the company, and then lets you ask follow-up questions about it — with real conversation memory, using GPT, Grok, or Groq's free tier.

In [20]:
%pip install tiktoken


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import tiktoken

In [ ]:
# Initialize and constants

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
groq_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("OpenAI API key looks good so far")
else:
    print("There might be a problem with your OpenAI API key? Please visit the troubleshooting notebook!")

openai = OpenAI(api_key=api_key)
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

MODEL = "gpt-4.1-mini"             
LINK_MODEL = "gpt-5-nano"          

encoding = tiktoken.get_encoding("cl100k_base")

OpenAI API key looks good so far


In [23]:
# Get all links from the OpenAI website

links = fetch_website_links("https://openai.com/")
links

['#main',
 '/',
 '/research/index/',
 '/business/',
 '/api/',
 '/about/',
 'https://openaifoundation.org',
 'https://chatgpt.com/',
 'https://openaifoundation.org',
 'https://chatgpt.com/',
 'https://chatgpt.com/?mode=voice',
 '/research/',
 '/api/',
 '/stories/',
 '/index/gpt-5-6/',
 '/index/gpt-5-6/',
 '/index/expanding-daybreak-as-the-cyber-defense-window-narrows/',
 '/index/improving-gpt-5-6-sol-in-chatgpt/',
 '/index/health-in-chatgpt/',
 '/news/company-announcements/',
 '/index/pacing-model-development-cyber-capabilities/',
 '/index/the-defenders-window/',
 '/index/dali-rajic-chief-revenue-officer/',
 '/index/advancing-the-price-performance-frontier-with-gpt-5-6/',
 '/index/introducing-openai-presence/',
 '/index/david-velez-robin-vince-join-openai-boards/',
 '/stories/',
 '/index/cycling-across-antarctica/',
 '/index/creating-new-simulations-black-holes/',
 '/index/chip-ganassi-racing/',
 '/research/index/',
 '/index/ten-advances-in-mathematics/',
 '/index/model-disproves-discre

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [24]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
Decide which links are most relevant to include in a brochure about the company,
such as an About page, Company page, or Careers/Jobs page.

IMPORTANT RULES:
- Only include links on the company's OWN domain. Ignore external sites such as
  LinkedIn, Twitter/X, GitHub, Discord, status pages, and external job boards.
- Do not include Terms of Service, Privacy, or email (mailto:) links.
- Return at most 5 links.
- Convert relative links such as "/about" into the full https URL.

Respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://full.url/careers"}
    ]
}
"""

In [25]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [26]:
print(get_links_user_prompt("https://openai.com/"))


Here is the list of links on the website https://openai.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
/
/research/index/
/business/
/api/
/about/
https://openaifoundation.org
https://chatgpt.com/
https://openaifoundation.org
https://chatgpt.com/
https://chatgpt.com/?mode=voice
/research/
/api/
/stories/
/index/gpt-5-6/
/index/gpt-5-6/
/index/expanding-daybreak-as-the-cyber-defense-window-narrows/
/index/improving-gpt-5-6-sol-in-chatgpt/
/index/health-in-chatgpt/
/news/company-announcements/
/index/pacing-model-development-cyber-capabilities/
/index/the-defenders-window/
/index/dali-rajic-chief-revenue-officer/
/index/advancing-the-price-performance-frontier-with-gpt-5-6/
/index/introducing-openai-presence/
/index/david-velez-robin-vince-join-openai-boards/
/stories/
/index/cycling-across-ant

In [27]:
def select_relevant_links(url):
    try:
        response = openai.chat.completions.create(
            model=LINK_MODEL,
            messages=[
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": get_links_user_prompt(url)}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Link selection failed: {e}")
        return {"links": []}

In [28]:
select_relevant_links("https://openai.com/")

{'links': [{'type': 'about page', 'url': 'https://openai.com/about/'},
  {'type': 'careers page', 'url': 'https://openai.com/careers/'},
  {'type': 'news page', 'url': 'https://openai.com/news/'},
  {'type': 'business page', 'url': 'https://openai.com/business/'},
  {'type': 'customer stories page',
   'url': 'https://openai.com/business/customer-stories/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [29]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links['links']:
        try:
            page_content = fetch_website_contents(link["url"])
            result += f"\n\n### Link: {link['type']}\n{page_content}"
        except Exception as e:
            print(f"Could not fetch {link['url']}: {e}")

    return result

In [30]:
print(fetch_page_and_all_relevant_links("https://openai.com/"))

## Landing Page:

OpenAI | Research & Deployment

Skip to main content
Research
Products
Business
Developers
Company
Foundation
(opens in a new window)
Log in
Try ChatGPT
(opens in a new window)
Research
Products
Business
Developers
Company
Foundation
(opens in a new window)
Try ChatGPT
(opens in a new window)
Login
OpenAI
What can I help with?
Message ChatGPT
Talk with ChatGPT
Research
API Platform
Stories
More
GPT-5.6: Frontier intelligence that scales with your ambition
Product
18 min read
GPT-5.6: Frontier intelligence that scales with your ambition
Product
18 min read
Expanding Daybreak as the Cyber Defense Window Narrows
Security
8 min read
Improving GPT‑5.6 Sol in ChatGPT—and expanding access to GPT-5.6 Luna for free users
Product
5 min read
Launching Health in ChatGPT
Product
Jul 23, 2026
7 min read
Recent news
View more
Pacing model development in an era of cyber-critical capabilities
Company
Aug 18, 2026
The Defender’s Window
Security
Aug 17, 2026
OpenAI appoints Dali Rajic a

In [31]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages
from a company website and creates a short brochure about the company.

The brochure is intended for:
- prospective customers
- investors
- potential employees

Include information such as:
- Company overview
- Products and services
- What makes the company different
- Customers or users
- Company culture
- Careers and job opportunities

Only use information provided in the website content.
Do not make up facts.

Respond in Markdown.
"""

In [32]:
MAX_TOKENS = 3_000  # keeps the prompt from getting too big or too expensive

def get_brochure_user_prompt(company_name, url):
    header = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    content = fetch_page_and_all_relevant_links(url)
    tokens = encoding.encode(content)[:MAX_TOKENS]
    print(f"Using {len(tokens)} tokens of content")
    return header + encoding.decode(tokens)

In [33]:
get_brochure_user_prompt("OpenAI", "https://openai.com")

Using 3000 tokens of content


'\nYou are looking at a company called: OpenAI\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nOpenAI | Research & Deployment\n\nSkip to main content\nResearch\nProducts\nBusiness\nDevelopers\nCompany\nFoundation\n(opens in a new window)\nLog in\nTry ChatGPT\n(opens in a new window)\nResearch\nProducts\nBusiness\nDevelopers\nCompany\nFoundation\n(opens in a new window)\nTry ChatGPT\n(opens in a new window)\nLogin\nOpenAI\nWhat can I help with?\nMessage ChatGPT\nTalk with ChatGPT\nResearch\nAPI Platform\nStories\nMore\nGPT-5.6: Frontier intelligence that scales with your ambition\nProduct\n18 min read\nGPT-5.6: Frontier intelligence that scales with your ambition\nProduct\n18 min read\nExpanding Daybreak as the Cyber Defense Window Narrows\nSecurity\n8 min read\nImproving GPT‑5.6 Sol in ChatGPT—and expanding access to GPT-5.6 Luna for free users\n

In [34]:
def create_brochure(company_name, url, client=openai, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))
    return result

In [ ]:
#create_brochure("OpenAI", "https://openai.com")


create_brochure("OpenAI", "https://openai.com", groq, "openai/gpt-oss-120b")

Using 2474 tokens of content


**OpenAI – AI Research, Products & Impact**  
*Empowering humanity with safe, beneficial artificial general intelligence (AGI)*  

---  

### Company Overview  
- **Mission:** Ensure that artificial general intelligence benefits all of humanity.  
- **Structure:** A public‑benefit corporation (OpenAI Group) governed by the nonprofit **OpenAI Foundation**.  
- **Vision:** Build safe, beneficial AGI while enabling others to achieve the same outcome.  
- **Core focus:** Frontier research on large language models, multimodal systems, reasoning, and safe deployment.  

---  

### flagship Products & Services  

| Product / Platform | Key Capabilities | Target Audience |
|--------------------|------------------|-----------------|
| **ChatGPT** (free, Business, Enterprise, Education) | Conversational AI, real‑time assistance, multimodal (text + images) | Individuals, schools, enterprises |
| **ChatGPT Work** | Enterprise controls, governance, integrated docs, decks & spreadsheets creation, background task automation | Large organizations seeking secure, scalable AI |
| **GPT‑5.6, GPT‑5.5, GPT‑5.4** | Frontier‑level language intelligence, higher price‑performance, available via API | Developers, startups, enterprises |
| **Codex** | Code generation and programming assistance | Developers & software teams |
| **API Platform** | Access to OpenAI models for custom applications, with detailed docs and SDKs | Developers, partners, SaaS providers |
| **Specialized Solutions** | *ChatGPT for Finance*, *Health in ChatGPT*, *ChatGPT for Education* | Industry verticals needing domain‑specific AI |
| **Research Tools** | GPT‑Rosalind (life‑sciences), visual models (ChatGPT Images 2.0), audio models (GPT‑Live, Voice Engine) | Researchers, scientists, academia |

---  

### What Makes OpenAI Different?  

- **Safety‑first charter:** A publicly disclosed charter and dedicated safety research guide every product launch.  
- **Public‑benefit governance:** The nonprofit foundation oversees the for‑profit group, aligning commercial success with societal good.  
- **Rapid frontier innovation:** Continuous releases (e.g., GPT‑5.6, multimodal visual/audio models) keep OpenAI at the cutting edge of AI capability.  
- **Enterprise‑ready controls:** ChatGPT Work adds governance, compliance, and integration tools for mission‑critical environments.  
- **Broad research impact:** Publications ranging from breakthroughs in mathematics to life‑science models demonstrate a deep commitment to advancing science.  

---  

### Customers & Users  

- **Enterprises:** Virgin Atlantic (customer‑journey optimization), Zapier (marketing automation), numerous SMBs and startups leveraging the API.  
- **Developers & Creators:** Codex users, API developers, and the broader developer community through the OpenAI Platform.  
- **Academic & Research Community:** Researchers using ChatGPT for academic projects, life‑science teams with GPT‑Rosalind, and institutions exploring multimodal AI.  
- **General Public:** Free‑tier ChatGPT users worldwide, benefiting from continuous feature upgrades (e.g., ChatGPT Images 2.0).  

---  

### Company Culture  

- **Humanity First:** Every decision is guided by the aim to uplift people and society.  
- **Humility:** OpenAI embraces iterative deployment, welcomes feedback, and acknowledges the limits of current knowledge.  
- **Feel the AGI:** A disciplined, imaginative approach to building transformative technology responsibly.  
- **Ship Joy:** Products are built to delight users and create positive change in daily life.  
- **Operating Principles:** “Find a way” – empowering teams to solve important problems with agency and collaboration.  

---  

### Careers & Opportunities  

- **Who we’re looking for:** Curious minds from **all disciplines** – AI research, engineering, product design, safety, policy, communications, and more.  
- **Values‑aligned hiring:** Candidates who champion humanity‑first thinking, humility, responsibility for AGI, and a joyful, optimistic outlook.  
- **Open roles:** Available on the **Careers** page – includes research scientists, safety engineers, product managers, business development, and operations.  
- **Why join:** Work on cutting‑edge AI, shape the future of technology, and contribute to a mission that prioritizes global benefit.  

---  

### Get In Touch  

- **Explore Products:** [OpenAI.com/products]  
- **Contact Sales / Partnerships:** Via the **Business** page (Contact sales button).  
- **Developer Resources:** Docs, SDKs, and community forums under **Developers**.  
- **Learn More:** Read the latest research, news stories, and safety approaches on the **Research** and **Company** sections.  

*Join OpenAI in building safe, beneficial AI that empowers humanity.*  

'**OpenAI – AI Research, Products & Impact**  \n*Empowering humanity with safe, beneficial artificial general intelligence (AGI)*  \n\n---  \n\n### Company Overview  \n- **Mission:** Ensure that artificial general intelligence benefits all of humanity.  \n- **Structure:** A public‑benefit corporation (OpenAI Group) governed by the nonprofit **OpenAI Foundation**.  \n- **Vision:** Build safe, beneficial AGI while enabling others to achieve the same outcome.  \n- **Core focus:** Frontier research on large language models, multimodal systems, reasoning, and safe deployment.  \n\n---  \n\n### flagship Products & Services  \n\n| Product / Platform | Key Capabilities | Target Audience |\n|--------------------|------------------|-----------------|\n| **ChatGPT** (free, Business, Enterprise, Education) | Conversational AI, real‑time assistance, multimodal (text\u202f+\u202fimages) | Individuals, schools, enterprises |\n| **ChatGPT Work** | Enterprise controls, governance, integrated docs, deck

In [37]:
def stream_brochure(company_name, url, client=openai, model=MODEL):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [38]:
brochure_text = stream_brochure("OpenAI", "https://openai.com")

Using 3000 tokens of content


# OpenAI Company Brochure

---

## Company Overview

OpenAI is a pioneering AI research and deployment company with a mission to ensure that artificial general intelligence (AGI)—AI systems that surpass human intelligence—benefits all of humanity. OpenAI operates with a vision focused on building safe, ethical, and beneficial AGI, aiming to either achieve this outcome or support others who do. The organization consists of two entities: the nonprofit OpenAI Foundation and the for-profit OpenAI Group, governed to collectively advance their public benefit mission.

---

## Products and Services

OpenAI delivers cutting-edge AI platforms and applications tailored to meet diverse business and individual needs:

- **ChatGPT**: A conversational AI assistant popular for daily problem solving, creative writing, education, and enterprise use.
- **ChatGPT Business & Enterprise**: Enhanced versions with governance controls for secure usage in professional settings.
- **API Platform**: Developers can integrate OpenAI’s models including GPT-5.6 for various applications such as customer service, marketing automation, research, and finance.
- **Codex**: AI tooling that assists with coding and software development.
- **ChatGPT Work**: Tools to automate tasks, generate polished documents, and support workflow efficiency integrated with preferred tools and plugins.

OpenAI continuously updates models with frontier intelligence like GPT-5.6, which pushes the boundaries of AI capabilities to scale with user ambitions.

---

## What Makes OpenAI Different

- **Mission-Driven Approach**: Prioritizes humanity-first AI development, focusing on safety, ethics, and broad societal benefit.
- **Research Excellence**: Produces leading research in theoretical computer science, mathematics, and AI safety.
- **Innovative and Iterative Development**: Incorporates humility by acknowledging knowledge limits and iteratively improving technologies using user and stakeholder feedback.
- **Cross-Disciplinary Talent**: Brings together experts from diverse fields to tackle the complexities of safe AGI.
- **Public Benefit Structure**: Combines nonprofit oversight with a public benefit corporation model to align profit incentives with social good.
- **Security & Privacy**: Maintains transparency and rigor in deployment to ensure users’ data and AI ethics are safeguarded.

---

## Customers and Users

OpenAI’s products serve a wide range of users including:

- **Individuals** using ChatGPT for education, creative writing, research, and personal productivity.
- **Startups and Small to Medium Businesses** leveraging AI to automate marketing, customer experiences, and core operations.
- **Large Enterprises** adopting ChatGPT Enterprise and API platforms for governance, security, and scaling AI across departments.
- **Researchers and Academics** accessing specialized AI tools designed for scientific exploration and innovation.
- **Developers and Partners** utilizing the API and Codex for building new AI-integrated applications.

Notable applications of OpenAI’s technology include enhancing customer journeys for Virgin Atlantic, automating finance work, and assisting in space research simulations.

---

## Company Culture

OpenAI fosters a unique culture rooted in core values that shape its approach to AI development:

- **Humanity First**: Committed to building AI that benefits people and society globally.
- **Act with Humility**: Embraces openness to new ideas and continuously adapts based on learnings.
- **Feel the AGI**: Recognizes the profound responsibility of AGI development, balancing creativity with discipline.
- **Ship Joy**: Aims to create joyful and transformational technology products.
- **Find a Way**: Encourages agency and determination to solve important problems.

This culture supports an optimistic yet responsible environment dedicated to impactful and ethical AI advancements.

---

## Careers and Job Opportunities

OpenAI welcomes curious, dedicated individuals from various backgrounds to join its mission to develop safe and beneficial AI systems. Career opportunities span multiple disciplines including AI research, engineering, policy, security, and product development. The company values diverse perspectives and a strong sense of responsibility towards humanity's future.

Current openings and detailed roles can be explored on their [Careers page](https://openai.com/careers).

---

## Contact and Further Information

- **Website:** [openai.com](https://openai.com)
- **Get Started with ChatGPT:** Available for free and subscription plans for business and enterprise.
- **For businesses:** Contact sales to integrate OpenAI’s AI platforms and tools for your organization.

---

OpenAI's commitment is clear: to advance AGI that helps humanity thrive, responsibly and innovatively. Join OpenAI’s journey to shape the future of technology today.

## New: chat about the company

Ask follow-up questions and get answers that remember what you already asked.

In [39]:
def chat_about_company(brochure_text, client=openai, model=MODEL):
    messages = [
        {"role": "system", "content": f"Answer questions about the company based only on this brochure:\n\n{brochure_text}"}
    ]
    print("Ask anything. Type 'quit' to stop.\n")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit"):
            break
        messages.append({"role": "user", "content": user_input})

        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
        reply = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            reply += chunk.choices[0].delta.content or ""
            update_display(Markdown(reply), display_id=display_handle.display_id)

        messages.append({"role": "assistant", "content": reply})

    return messages

In [40]:
conversation = chat_about_company(brochure_text)

Ask anything. Type 'quit' to stop.



OpenAI’s products serve a diverse range of customers and users, including:

- **Individuals** who use ChatGPT for purposes such as education, creative writing, research, and enhancing personal productivity.
- **Startups and Small to Medium Businesses** that leverage AI to automate marketing, improve customer experiences, and streamline core operations.
- **Large Enterprises** that adopt ChatGPT Enterprise and API platforms for needs related to governance, security, and scaling AI applications across various departments.
- **Researchers and Academics** who utilize specialized AI tools designed for scientific exploration and innovation.
- **Developers and Partners** who use OpenAI’s API and Codex tools to build AI-integrated applications.

Notable uses of OpenAI’s technology include enhancing the customer journey for Virgin Atlantic, automating financial tasks, and supporting space research simulations.

OpenAI is a pioneering AI research and deployment company with the mission to ensure that artificial general intelligence (AGI)—AI systems that surpass human intelligence—benefits all of humanity. The company operates under a vision focused on building safe, ethical, and beneficial AGI, either by achieving this goal themselves or by supporting others in the effort.

OpenAI consists of two entities: the nonprofit OpenAI Foundation and the for-profit OpenAI Group. These are governed together to advance their public benefit mission.

The company is dedicated to developing AI technologies responsibly, with strong emphasis on safety, ethics, and societal benefit. They bring together experts from various disciplines to tackle the complexity of safe AGI development and maintain transparency and rigorous security and privacy standards to protect users.

OpenAI offers a suite of advanced AI products and platforms including ChatGPT for conversational AI, enhanced enterprise versions, developer APIs like GPT-5.6, AI coding tools such as Codex, and productivity automation tools under ChatGPT Work.

Their culture centers on values like putting humanity first, acting with humility, balancing the responsibility of AGI development, creating joyful technology, and fostering determination to solve vital problems.

Overall, OpenAI is committed to innovating with AI that helps humanity thrive in a responsible and ethical manner.